In [ ]:
import pandas as pd

# Define paths to the saved feature files
mocap_features_path = '/content/drive/MyDrive/mocap_features.csv'
radar_features_path = '/content/drive/MyDrive/radar_features.csv'
fp_features_path    = '/content/drive/MyDrive/fp_features.csv'
output_merged_path  = '/content/drive/MyDrive/OLST_Final_Merged.csv'
olst_attempts_path  = '/content/501Project_Dataset/multimodal-synchronized-motion-capture-force-plate-and-radar-dataset-of-the-one-legged-stand-test-for-fall-risk-assessment-1.0/Metadata/OLST_Attempts.csv'

# Load the feature DataFrames
try:
    mocap_df = pd.read_csv(mocap_features_path)
    print(f'Loaded MOCAP features: {len(mocap_df)} rows')
except FileNotFoundError:
    print(f'Error: MOCAP features file not found at {mocap_features_path}')
    mocap_df = pd.DataFrame()

try:
    radar_df = pd.read_csv(radar_features_path)
    print(f'Loaded RADAR features: {len(radar_df)} rows')
except FileNotFoundError:
    print(f'Error: RADAR features file not found at {radar_features_path}')
    radar_df = pd.DataFrame()

try:
    fp_df = pd.read_csv(fp_features_path)
    print(f'Loaded FORCE PLATE features: {len(fp_df)} rows')
except FileNotFoundError:
    print(f'Error: FORCE PLATE features file not found at {fp_features_path}')
    fp_df = pd.DataFrame()

# Load the original OLST_Attempts.csv for metadata
try:
    olst_df_raw = pd.read_csv(olst_attempts_path)
    print(f'Loaded OLST_Attempts.csv: {len(olst_df_raw)} rows')
    # Pre-process raw olst_df for merging:
    # Extract participant_id for consistency (will be dropped from raw_df before final merge)
    olst_df_raw['participant_id'] = olst_df_raw['OLST_attempt_id'].str.slice(0, 2).astype(int)
    # Classify outcome (label) for consistency, though 'label' from features is preferred
    olst_df_raw['label'] = olst_df_raw['t_stable'].apply(lambda x: 'STABLE' if pd.notna(x) else 'UNSTABLE')
except FileNotFoundError:
    print(f'Error: OLST_Attempts.csv not found at {olst_attempts_path}')
    olst_df_raw = pd.DataFrame()

# Define common columns for merging feature dataframes
# These columns should be identical across mocap, radar, fp dataframes for an inner merge
feature_common_cols = ['OLST_attempt_id', 'participant_id', 'stance_leg', 'lifted_leg', 'label']

# Start with olst_df_raw as base and perform LEFT merges to keep all rows
all_features_df = pd.DataFrame()
if not olst_df_raw.empty:
    all_features_df = olst_df_raw.copy()

    if not mocap_df.empty:
        # Ensure 'stance_leg' and 'lifted_leg' are in olst_df_raw if needed for merging
        # These columns are derived during feature extraction, so we need to add them to olst_df_raw for consistent merging keys.
        # This derivation is implicitly happening when features are extracted, so we must add it to the base df.
        # Let's re-add the movement_code derivation here or assume it's already present in olst_df_raw for left merge keys.
        # For this scenario, it's safer to merge with feature_common_cols that are present in the feature DFs.
        # We should only add metadata from olst_df_raw that isn't already part of the feature set.

        # First, ensure that derived columns for merging keys are consistent or present in olst_df_raw if we use it as base.
        # The feature DFs already have 'stance_leg', 'lifted_leg', 'label'.
        # We need to merge olst_df_raw onto the *merged features* or carefully ensure all keys exist.

        # A simpler approach: merge feature DFs first, then left merge with olst_df_raw for all metadata.
        # Resetting the merge logic to build feature DF then left merge metadata.

        # Re-defining feature merge logic to preserve OLST_Attempts.csv rows
        print("\n--- Merging features with OLST_Attempts.csv as base ---")
        # First, ensure olst_df_raw has the necessary 'stance_leg' and 'lifted_leg' for consistent merging keys if they're not in the feature_common_cols from olst_df_raw
        # They are derived in feature extraction, so we need them for merging. We must derive them for olst_df_raw as well.

        # Re-derive 'movement_code', 'stance_leg', 'lifted_leg' for olst_df_raw
        olst_df_raw['movement_code'] = olst_df_raw['RADAR_capture'].str.split('_').str[1]
        olst_df_raw['stance_leg'] = olst_df_raw['movement_code'].str[-1]
        olst_df_raw['lifted_leg'] = olst_df_raw['stance_leg'].apply(lambda s: 'R' if s == 'L' else 'L')

        all_features_df = olst_df_raw.copy() # Start with all attempts from olst_df_raw

        if not mocap_df.empty:
            all_features_df = pd.merge(all_features_df, mocap_df, on=feature_common_cols, how='left', suffixes=('', '_mocap'))
            print(f'Left merged with MOCAP features. Current rows: {len(all_features_df)}')

        if not radar_df.empty:
            all_features_df = pd.merge(all_features_df, radar_df, on=feature_common_cols, how='left', suffixes=('', '_radar'))
            print(f'Left merged with RADAR features. Current rows: {len(all_features_df)}')

        if not fp_df.empty:
            all_features_df = pd.merge(all_features_df, fp_df, on=feature_common_cols, how='left', suffixes=('', '_fp'))
            print(f'Left merged with FORCE PLATE features. Current rows: {len(all_features_df)}')

        # Clean up duplicate columns that might arise from merging 'label', 'participant_id' and 'movement_code'
        # when they are also present in the feature dataframes. We prioritize the ones from olst_df_raw or features as they are consistent.
        # The feature_common_cols are used as merge keys, so their original versions are preserved.

        # Drop the redundant 'movement_code_radar', 'movement_code_fp', etc., if they were created and are duplicates of the 'movement_code' from olst_df_raw
        # Or, specifically drop the derived cols we added to `olst_df_raw` if the feature DFs also provide them as primary keys.
        # Since we use `olst_df_raw` as base, its `stance_leg`, `lifted_leg`, `label`, `participant_id` are the ones kept.
        # The suffixes are for non-key columns.

        # A simpler way to handle this: ensure no duplicate columns are formed by carefully selecting columns from olst_df_raw.
        # After all left merges, the original columns from olst_df_raw will be present.
        # If feature DFs have columns like `participant_id`, `label`, etc., they were used in `on=` clause and are not duplicated.
        # So the suffix mechanism for non-key columns is working as expected.

else:
    print('OLST_Attempts DataFrame is empty, cannot proceed with merge.')


if not all_features_df.empty:
    print(f'\nFinal merged dataset shape: {all_features_df.shape}')
    print('\nFirst 5 rows of the merged dataset:')
    display(all_features_df.head())

    # Save the merged DataFrame
    all_features_df.to_csv(output_merged_path, index=False)
    print(f'\n✓ Merged features saved to: {output_merged_path}')
else:
    print('\nNo data to merge or save.')